In [1]:
import os
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
import librosa
import numpy as np
from collections import Counter

### Training SVM Model

In [2]:
# The number of seconds to use for each audio clip when extracting features and training the model. Either 3 or 30.
NUMBER_OF_SECONDS = 3
# Columns not taken into consideration for training the model, as they are not relevant for genre classification or are redundant.
COLUMNS_TO_DROP = ['filename', 'label', 'length', 'tempo','harmony_mean', 'harmony_var', 'perceptr_var', 'perceptr_mean']

In [3]:
from src.extract_features import extract_features

def train_model(csv_path: str):
    df = pd.read_csv(csv_path)

    X = df.drop(columns=COLUMNS_TO_DROP)
    y = df["label"]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    svm_clf = Pipeline([
        ("scaler", StandardScaler()),
        ("svm", SVC(kernel="rbf", C=10, gamma=0.01))
    ])

    svm_clf.fit(X_train, y_train)

    return svm_clf

def load_preprocess_and_segment(
    filepath: str,
    sr: int = 22050,
    clip_seconds: int = 30,
    segment_seconds: int = 3,
    offset_seconds: float = 0.0,
    normalize: str = "peak",  # "peak", "rms", or None
):
    """
    Loads an audio file (mp3/wav/etc.), converts to mono, resamples to sr,
    takes exactly `clip_seconds` seconds starting at `offset_seconds`,
    and splits into fixed segments of `segment_seconds`.

    Returns:
        y_clip: np.ndarray of shape (clip_seconds * sr,)
        segments: list[np.ndarray] length = clip_seconds / segment_seconds
        sr: int
    """

    if not os.path.exists(filepath):
        raise FileNotFoundError(
            f"File not found:\n{filepath}\n\n"
            "Tip: 'search-ms:...' paths are not real file paths. "
            "Use the actual file path like C:\\Users\\...\\Downloads\\file.mp3"
        )

    # 1) Load audio (forces mono + resamples to sr)
    # duration=clip_seconds loads ONLY the first 30 seconds (fast & consistent)
    y, sr_loaded = librosa.load(
        filepath,
        sr=sr,
        mono=True,
        offset=offset_seconds,
        duration=clip_seconds
    )

    # 2) Ensure exact length (pad with zeros if shorter, cut if longer)
    target_len = int(clip_seconds * sr)
    if len(y) < target_len:
        y = np.pad(y, (0, target_len - len(y)), mode="constant")
    else:
        y = y[:target_len]

    # 3) Remove DC offset (small bias) - good hygiene
    y = y - np.mean(y)

    # 4) Normalize (recommended)
    if normalize == "peak":
        peak = np.max(np.abs(y)) + 1e-12
        y = y / peak
    elif normalize == "rms":
        rms = np.sqrt(np.mean(y**2)) + 1e-12
        y = y / rms
    elif normalize is None:
        pass
    else:
        raise ValueError("normalize must be 'peak', 'rms', or None")

    # 5) Segment into fixed windows
    seg_len = int(segment_seconds * sr)
    n_segments = clip_seconds // segment_seconds  # 30//3 = 10

    segments = []
    for i in range(n_segments):
        start = i * seg_len
        end = start + seg_len
        segments.append(y[start:end])

    return y, segments, sr

def predict_song_genre(svm_clf, df_segments: pd.DataFrame):
    """
    Predict genre for each segment and return majority vote + full list.
    """
    seg_preds = svm_clf.predict(df_segments)
    final = Counter(seg_preds).most_common(1)[0][0]
    return final, seg_preds

In [4]:
import warnings
warnings.filterwarnings("ignore")

# ---------- Paths ----------
csv_path = f"../data/features_{NUMBER_OF_SECONDS}_sec.csv"
audio_path = r"../data/custom_songs/still_dre.mp3"  # <-- Change this to your song path

# ---------- 1) Load trained model (or train once) ----------
svm_clf = train_model(csv_path)

# ---------- 2) Load training CSV to get expected feature order ----------
df_train = pd.read_csv(csv_path)
expected_columns = list(df_train.drop(columns=COLUMNS_TO_DROP).columns)

# ---------- 3) Extract features from the song ----------
df_segments = extract_features(audio_path, segment_duration=NUMBER_OF_SECONDS)

print("Segments:", len(df_segments))

# ---------- 4) Prepare features for prediction ----------
df_segments = df_segments.drop(columns=COLUMNS_TO_DROP)
df_segments = df_segments[expected_columns]

# ---------- 5) Predict ----------
final_genre, segment_preds = predict_song_genre(svm_clf, df_segments)

print("\nSegment predictions:")
print(segment_preds)

print("\nFinal predicted genre:")
print(final_genre)


Segments: 97

Segment predictions:
['classical' 'classical' 'classical' 'classical' 'classical' 'rock' 'jazz'
 'jazz' 'hiphop' 'hiphop' 'reggae' 'hiphop' 'hiphop' 'reggae' 'hiphop'
 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop'
 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop'
 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop'
 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop'
 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop'
 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop'
 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop'
 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop'
 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'hiphop' 'reggae'
 'reggae' 'reggae' 'reggae' 'reggae' 'reggae' 'reggae' 'pop' 'reggae'
 'reggae' 'hiphop']

Final predicted genre:
hiphop


In [5]:
import joblib

SVM = {
    "model": svm_clf,
}
joblib.dump(SVM, f"../models/SVM_{NUMBER_OF_SECONDS}_sec.joblib")

['../models/SVM_3_sec.joblib']